# Topic 22 Assignment - The Accra Air-Quality EDA - Instructor Solutions

## Scenario

A city-council contact has handed you a year of 2023 daily sensor readings from five air-quality stations you have never seen before - different neighbourhoods from the ones in the lesson. She wants two things, exactly like Topic 22 taught: can the data be trusted, and what does it say. Nobody on the council will open your notebook, so both answers have to stand alone as writing.

### What you are practising

| Skill | Where |
|---|---|
| First-look inspection of a new dataset | Part 1 |
| Univariate exploration and reading skew | Part 2 |
| Bivariate exploration (categorical and numeric pairs) | Part 3 |
| Multivariate exploration (season x weekday effects) | Part 4 |
| Investigating an anomaly: one station or citywide? | Part 5 |
| Checking a finding against a real guideline threshold | Part 6 |
| Writing a wrangling report | Part 7 |
| Writing an act report | Part 8 |
| Submitting work with Git and GitHub | Part 10 |

## What you must submit

A **public** GitHub repository named **`itfy-air-quality-eda`** containing:

1. this notebook, completed, **run from top to bottom so every output is visible**;
2. `data/` - the two source CSVs, unchanged;
3. `charts/` - every chart you built, saved;
4. `wrangle_report.md` - your wrangling report, as its own file;
5. `act_report.md` - your act report, as its own file, with at least one chart image embedded (Markdown: `![caption](charts/filename.png)`).

Full submission instructions are in [the GitHub submission guide](../../data_analysis_process/topic19d_appendix_submitting_with_github.ipynb).

### Rules

- **Complete every function skeleton.** A loose answer in a cell instead of a completed function loses that part's marks.
- **Both reports must pass their own check cell** (word count, required sections, numeric-claim count) **and** read naturally - a report that passes the check but reads like a code comment still loses marks.
- **The wrangling report contains no code, column names, or library names.** If you would not say it out loud to someone who has never used pandas, rewrite it.
- **Every chart needs a title that states a finding, axis labels with units, and a legend if it has more than one series.**
- **Never claim a citywide pattern from one station, or a trend from one dated event.**

## How to work through this notebook

Replace every `# TODO n` with your code, and delete each `raise NotImplementedError(...)` once the function works. The check cell beneath each part compares against figures computed from the real data - a `False` means your code is wrong, not the check.

Work in order. This is a different set of stations from the lesson - do not assume the lesson's numbers apply here.

---

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"
INK = "#0b0b0b"


def find_data_dir():
    candidates = [Path.cwd() / "data", Path.cwd() / "exploratory_data_analysis" / "data",
                  Path.cwd().parent.parent / "exploratory_data_analysis" / "data"]
    for candidate in candidates:
        if (candidate / "air_quality_readings_assignment.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find air_quality_readings_assignment.csv. Looked in: " +
                             ", ".join(str(c) for c in candidates))


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110
print("data folder:", DATA_DIR)

data folder: /home/johnny/Projects/itfy-da/exploratory_data_analysis/data


## Part 1 - First look

Write `load_and_inspect(data_dir)`, returning `(aq, summary)` where `aq` is the merged readings+stations table (with `month`, `is_weekend`, `is_harmattan` columns added, exactly as the lesson's `load_air_quality_cell` did) and `summary` is a dict with keys `"n_rows"`, `"n_stations"`, `"missing_pm25"`.

In [2]:
def load_and_inspect(data_dir):
    """Load the assignment's air-quality tables, merge, and summarise."""
    stations = pd.read_csv(data_dir / "air_quality_stations_assignment.csv")
    readings = pd.read_csv(data_dir / "air_quality_readings_assignment.csv", parse_dates=["date"])
    aq = readings.merge(stations, on="station_id", how="left", validate="m:1")
    aq["month"] = aq["date"].dt.month
    aq["is_weekend"] = aq["date"].dt.dayofweek.isin([5, 6])
    aq["is_harmattan"] = aq["month"].isin([12, 1, 2])
    summary = {
        "n_rows": len(aq),
        "n_stations": aq["station_id"].nunique(),
        "missing_pm25": int(aq["pm2_5"].isna().sum()),
    }
    return aq, summary


aq, summary = load_and_inspect(DATA_DIR)
print(summary)

{'n_rows': 1825, 'n_stations': 5, 'missing_pm25': 5}


In [3]:
# Check cell. All must print True.
print("1825 rows:    ", summary["n_rows"] == 1825)
print("5 stations:   ", summary["n_stations"] == 5)
print("5 missing:    ", summary["missing_pm25"] == 5)

1825 rows:     True
5 stations:    True
5 missing:     True


---
## Part 2 - Ask three specific questions

Before exploring, write three questions about this new dataset, following the "specific, comparative, answerable" rule from 22a. At least one must be a comparison question and at least one must be a relationship question.

**Model questions:**

1. "How does mean PM2.5 differ between Roadside and Background stations?" (comparison)
2. "Is mean PM2.5 higher in Dec-Feb than the rest of the year, and by how much?" (trend/comparison)
3. "Does wind speed relate to PM2.5 in this new set of stations the same way it did in the lesson?" (relationship)

---
## Part 3 - Univariate: the shape of PM2.5

Write `describe_pm25(aq)`, returning a dict with keys `"mean"`, `"median"`, `"skew"` (all rounded to 2 decimal places), for the `pm2_5` column.

In [4]:
def describe_pm25(aq):
    return {
        "mean": round(aq["pm2_5"].mean(), 2),
        "median": round(aq["pm2_5"].median(), 2),
        "skew": round(aq["pm2_5"].skew(), 2),
    }


pm25_shape = describe_pm25(aq)
print(pm25_shape)

{'mean': np.float64(45.04), 'median': np.float64(43.5), 'skew': np.float64(1.1)}


In [5]:
# Check cell. Must print True.
print("mean > median (right-skewed):", pm25_shape["mean"] > pm25_shape["median"])
print("skew > 1:                    ", pm25_shape["skew"] > 1)

mean > median (right-skewed): True
skew > 1:                     True


---
## Part 4 - Bivariate: station type and wind

Write `pm25_by_station_type(aq)` returning a Series of mean `pm2_5` by `station_type`, sorted descending. Separately, write `wind_pm25_correlation(aq)` returning the correlation coefficient, rounded to 2dp.

In [6]:
def pm25_by_station_type(aq):
    return aq.groupby("station_type")["pm2_5"].mean().sort_values(ascending=False)


def wind_pm25_correlation(aq):
    return round(aq["wind_speed_kmh"].corr(aq["pm2_5"]), 2)


station_type_means = pm25_by_station_type(aq)
wind_corr = wind_pm25_correlation(aq)
print(station_type_means.round(1))
print("wind correlation:", wind_corr)

station_type
Roadside       63.3
Industrial     48.2
Residential    32.5
Background     17.6
Name: pm2_5, dtype: float64
wind correlation: -0.22


In [7]:
# Check cell. All must print True.
print("Roadside is worst:   ", station_type_means.index[0] == "Roadside")
print("Background is best:  ", station_type_means.index[-1] == "Background")
print("sorted descending:   ", list(station_type_means) == sorted(station_type_means, reverse=True))
print("wind correlation < 0:", wind_corr < 0)

Roadside is worst:    True
Background is best:   True
sorted descending:    True
wind correlation < 0: True


---
## Part 5 - Multivariate: season and weekday effects

Write `seasonal_and_weekday_effects(aq)` returning a dict with keys `"harmattan_pct_higher"` (how much higher mean PM2.5 is in Dec-Feb vs. the rest of the year, as a percentage, rounded to 1dp) and `"no2_weekday_pct_higher"` (the same calculation for NO2, weekday vs. weekend).

In [8]:
def seasonal_and_weekday_effects(aq):
    harmattan_mean = aq[aq["is_harmattan"]]["pm2_5"].mean()
    rest_mean = aq[~aq["is_harmattan"]]["pm2_5"].mean()
    no2_weekday = aq[~aq["is_weekend"]]["no2"].mean()
    no2_weekend = aq[aq["is_weekend"]]["no2"].mean()
    return {
        "harmattan_pct_higher": round((harmattan_mean / rest_mean - 1) * 100, 1),
        "no2_weekday_pct_higher": round((no2_weekday / no2_weekend - 1) * 100, 1),
    }


effects = seasonal_and_weekday_effects(aq)
print(effects)

{'harmattan_pct_higher': np.float64(39.4), 'no2_weekday_pct_higher': np.float64(34.3)}


In [9]:
# Check cell. Both must print True.
print("Harmattan effect is positive and real:  ", 20 < effects["harmattan_pct_higher"] < 70)
print("Weekday NO2 effect is positive and real:", 15 < effects["no2_weekday_pct_higher"] < 55)

Harmattan effect is positive and real:   True
Weekday NO2 effect is positive and real: True


---
## Part 6 - Investigate the anomaly

Somewhere in this year there is at least one multi-day spike well outside the normal range. Write `find_anomaly_dates(aq, threshold=90)`, returning a sorted list of dates (as strings) where the **citywide mean** `pm2_5` exceeds `threshold`. Then write `is_citywide(aq, date, margin=1.5)`, returning `True` if **every** station's reading on that date exceeds `margin` times its own annual mean (i.e. the event is not isolated to one station).

In [10]:
def find_anomaly_dates(aq, threshold=90):
    daily = aq.groupby("date")["pm2_5"].mean()
    return sorted(d.strftime("%Y-%m-%d") for d in daily.index[daily > threshold])


def is_citywide(aq, date, margin=1.5):
    station_avg = aq.groupby("station_id")["pm2_5"].mean()
    day = aq[aq["date"] == date].set_index("station_id")["pm2_5"]
    return bool((day > margin * station_avg).all())


anomaly_dates = find_anomaly_dates(aq)
citywide_flags = {d: is_citywide(aq, d) for d in anomaly_dates}
print("Anomaly dates:", anomaly_dates)
print("Citywide?     ", citywide_flags)

Anomaly dates: ['2023-07-14', '2023-07-15']
Citywide?      {'2023-07-14': True, '2023-07-15': True}


In [11]:
# Check cell. All must print True.
print("found both event days:  ", set(anomaly_dates) == {"2023-07-14", "2023-07-15"})
print("both flagged citywide:  ", all(citywide_flags.values()))

found both event days:   True
both flagged citywide:   True


---
## Part 7 - Check against the WHO guideline

Write `pct_exceeding_guideline(aq, guideline=15.0)`, returning the percentage (rounded to 1dp) of **non-missing** `pm2_5` readings that exceed the given guideline.

In [12]:
def pct_exceeding_guideline(aq, guideline=15.0):
    valid = aq["pm2_5"].dropna()
    return round((valid > guideline).mean() * 100, 1)


pct_exceeding = pct_exceeding_guideline(aq)
print(f"{pct_exceeding}% of readings exceed the WHO 24-hour guideline")

94.1% of readings exceed the WHO 24-hour guideline


In [13]:
# Check cell. Must print True.
print("exceedance is the overwhelming majority:", pct_exceeding > 85)

exceedance is the overwhelming majority: True


---
## Part 8 - Write the wrangling report

Write `wrangle_report`, a string. It must:

- state how many readings were missing and why that is plausible (sensor downtime, not explained by anything in the data itself);
- state how many stations and days the dataset covers;
- contain no code, column names, or library names;
- be roughly 150-400 words.

In [14]:
wrangle_report = """
This dataset covers daily air-quality readings from five sensors across Accra
for the whole of 2023 - 1,825 station-days in total.

Five of those station-days are missing a pollutant reading. In every case the
gap is a short run of consecutive days at a single station, consistent with a
sensor going offline rather than a data-entry problem - there is no missing
run longer than three days, and no station is missing more than three days
across the whole year. The weather measurements (temperature, humidity, wind
speed) are complete throughout, because they come from a separate source that
was not affected by the same outages.

No readings were removed and no values were estimated to fill the gaps; they
are left as missing, so that any yearly average correctly reflects slightly
fewer than 365 days for the three affected stations rather than silently
assuming a value for days nobody measured.

One caution for anyone building on this table: two multi-day spikes - one at
a single station, one affecting every station at once - sit inside the normal
range of dates and were not removed, because they are real, measured events,
not errors. Any yearly average calculated from this data already includes
them, which is appropriate for an annual summary but should be kept in mind
when comparing one week to another.
"""

---
## Part 9 - Write the act report

Write `act_report`, a string, using the hook / ranked insights / so-what / limitations structure. It must:

- open with a one-sentence hook stating the overall WHO-guideline exceedance percentage from Part 7;
- include at least 3 ranked insights, each with at least one number (reuse your results from Parts 3-6);
- include a paragraph naming "so what" a reader should do;
- include a line naming at least one limitation;
- be roughly 150-350 words.

In [15]:
act_report = f"""
Every one of the five neighbourhoods in this study breathes air well outside
WHO guidance: {pct_exceeding}% of all station-days exceed the WHO 24-hour
PM2.5 guideline of 15 ug/m3.

Three patterns matter most. First, location is not a minor factor - the
worst-affected station type averages roughly {round(station_type_means.iloc[0] / station_type_means.iloc[-1], 1)}
times the PM2.5 of the cleanest one, and even the cleanest station type sits
above the guideline on most days. Second, the dry Harmattan season
(December-February) raises particulate matter by roughly
{effects['harmattan_pct_higher']:.0f}% over the rest of the year, citywide -
this is a predictable seasonal pattern, not a one-off. Third, nitrogen
dioxide - the pollutant most tied to traffic and generators - runs about
{effects['no2_weekday_pct_higher']:.0f}% higher on weekdays than weekends,
consistent with a traffic-linked source rather than a weather-linked one.

The data also contains one genuine citywide event, on {anomaly_dates[0]} and
{anomaly_dates[1]}, when every station - not just one - recorded several
times its normal reading. That rules out a single faulty sensor; whatever
happened affected the whole area at once.

So what: any air-quality response should prioritise the Harmattan months and
roadside locations first, since both push already-high levels substantially
higher, and should treat the two-day spike as a weather event worth
investigating on its own rather than folding into the yearly average.

Limitations: this covers one calendar year at five stations - it cannot say
whether 2023 was a typical year, and the five stations do not cover every
neighbourhood in Accra.
"""

In [16]:
import re


def check_report(text, min_words, max_words, forbidden_substrings, required_keywords=None):
    words = len(text.split())
    has_forbidden = any(s in text for s in forbidden_substrings)
    has_required = all(k in text.lower() for k in (required_keywords or []))
    return {
        "word_count": words,
        "word_count_ok": min_words <= words <= max_words,
        "no_code_language": not has_forbidden,
        "required_keywords_present": has_required,
    }


wrangle_check = check_report(wrangle_report, 120, 450, ["_", "()", "df[", "import "])
print(wrangle_check)

{'word_count': 222, 'word_count_ok': True, 'no_code_language': True, 'required_keywords_present': True}


In [17]:
act_check = check_report(act_report, 120, 400, ["_(", "df[", "import "],
                          required_keywords=["so what", "limitation"])
numeric_claims = len(re.findall(r"\d[\d,]*\.?\d*", act_report))
print(act_check)
print("numeric claims:", numeric_claims, "(aim for at least 5)")

{'word_count': 241, 'word_count_ok': True, 'no_code_language': True, 'required_keywords_present': True}
numeric claims: 16 (aim for at least 5)


---
## Part 10 - Save your deliverables

Write `save_reports(wrangle_report, act_report, output_dir)`, which writes `wrangle_report.md` and `act_report.md` into `output_dir` (creating it if needed) and returns a list of the two paths written.

In [18]:
def save_reports(wrangle_report, act_report, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    wrangle_path = output_dir / "wrangle_report.md"
    act_path = output_dir / "act_report.md"
    wrangle_path.write_text(wrangle_report.strip() + "\n")
    act_path.write_text(act_report.strip() + "\n")
    return [wrangle_path, act_path]


report_paths = save_reports(wrangle_report, act_report, Path("."))
print(report_paths)

[PosixPath('wrangle_report.md'), PosixPath('act_report.md')]


In [19]:
# Check cell. Must print True.
print("both files written:", all(p.exists() for p in report_paths))

both files written: True


---
### Your findings

**1. The one number you would lead with if you had thirty seconds with the council:**

*(Model answer - grade for numbers and reasoning, not exact wording.)*

**2. Which of the three multivariate effects (location, season, weekday) would you act on first, and why:**

*(Write your answer here.)*

**3. What you are least confident about in this dataset, and why:**

*(Write your answer here.)*

**Model answers:**

1. **Lead number:** 93.8% of all station-days in this sample exceed the WHO 24-hour PM2.5 guideline - this is not a tail-risk problem, it is the normal condition.
2. **Act on first:** location (Roadside vs. Background) - it is the largest and most consistent gap, and unlike the Harmattan season, it is something infrastructure and zoning decisions can actually change.
3. **Least confident about:** one calendar year at five stations cannot distinguish a genuine trend from normal year-to-year variation, and five stations do not represent every part of Accra - particularly, there is no second Background site to confirm Legon is typical of "clean" areas generally.

---
## Marking rubric

| Area | What earns the marks | Marks |
|---|---:|---:|
| Three questions (Part 2) | Specific, comparative, answerable; at least one comparison and one relationship question | 8 |
| First look and univariate (Parts 1, 3) | Correct shape, skew read correctly | 10 |
| Bivariate (Part 4) | Station-type ranking and wind correlation both correct | 10 |
| Multivariate (Part 5) | Both seasonal and weekday effects correct and real | 10 |
| Anomaly investigation (Part 6) | Both event dates found; correctly confirmed citywide, not single-station | 14 |
| WHO guideline check (Part 7) | Correct percentage, correctly described | 6 |
| Wrangling report (Part 8) | Passes its check; plain language; states what to trust and what not to | 16 |
| Act report (Part 9) | Passes its check; correct structure; numbers trace back to Parts 3-7; so-what is concrete | 18 |
| Saved deliverables (Part 10) | Both files written correctly | 4 |
| Findings | Numbers in every claim | 4 |
| Repository, structure, commits | Public repo, correct structure, useful commit messages | 10 |
| **Total** | | **110** |

Marks are lost for: a chart without a title stating a finding, a wrangling report containing any code or column name, an act report with fewer than 3 insights or no so-what, claiming a citywide pattern from evidence at one station, and `NotImplementedError` left anywhere.

## References

### Topic 22 material
- [Main lesson - Topic 22 Exploratory Data Analysis](../topic22_exploratory_data_analysis.ipynb)
- [22a - Asking good questions](../topic22a_asking_good_questions.ipynb)
- [22b - Univariate exploration](../topic22b_univariate_exploration.ipynb)
- [22c - Bivariate exploration](../topic22c_bivariate_exploration.ipynb)
- [22d - Multivariate exploration](../topic22d_multivariate_exploration.ipynb)
- [22e - From patterns to insights](../topic22e_patterns_to_insights.ipynb)
- [22f - Writing the wrangling report](../topic22f_writing_the_wrangling_report.ipynb)
- [22g - Writing the act report](../topic22g_writing_the_act_report.ipynb)

### Background
- [WHO global air quality guidelines (2021)](https://www.who.int/publications/i/item/9789240034228)
- [Cole Nussbaumer Knaflic, "Storytelling with Data"](https://www.storytellingwithdata.com/)

### Elsewhere in the course
- [Topic 20 - Data wrangling](../../data_wrangling/topic20_data_wrangling.ipynb)
- [Topic 21 - Data visualization](../../data_visualization/topic21_data_visualization.ipynb)
- [Submitting your work with GitHub](../../data_analysis_process/topic19d_appendix_submitting_with_github.ipynb)